In [1]:
# need to add path using os and sys first
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../")))

In [2]:
from models.embedding import StableEmbedding

In [3]:
import torch

In [4]:
torch.__version__

'2.1.0+cu121'

In [5]:
batch_size = 2
embedding_size = 8
num_heads = 2
seq_len = 3
query_seq_len = 4
vocab_size = 4

In [6]:
dtype = torch.float32
device = "cuda"

In [7]:
q_idx = torch.randint(0, vocab_size, (batch_size, query_seq_len))
kv_idx = torch.randint(0, vocab_size, (batch_size, seq_len))

In [8]:
q_idx = q_idx.to(device)
kv_idx = kv_idx.to(device)

In [9]:
emb = StableEmbedding(vocab_size, embedding_size)
emb = emb.to(device)

In [10]:
q = emb(q_idx)
kv = emb(kv_idx)

In [11]:
q_proj = torch.nn.Linear(embedding_size, embedding_size)
kv_proj = torch.nn.Linear(embedding_size, embedding_size)

In [12]:
q_proj = q_proj.to(device)
kv_proj = kv_proj.to(device)

In [13]:
q_heads = q_proj(q).view(batch_size, query_seq_len, num_heads, embedding_size // num_heads).transpose(1, 2)
kv_heads = kv_proj(kv).view(batch_size, seq_len, num_heads, embedding_size // num_heads).transpose(1, 2)

In [14]:
q_heads.shape, kv_heads.shape

(torch.Size([2, 2, 4, 4]), torch.Size([2, 2, 3, 4]))

In [17]:
mask = torch.ones(query_seq_len, seq_len, dtype=dtype, device=device)

In [18]:
with torch.backends.cuda.sdp_kernel(enable_flash=False, enable_math=False, enable_mem_efficient=True):
    out = torch.nn.functional.scaled_dot_product_attention(q_heads, kv_heads, kv_heads, mask)

In [20]:
out.shape # (batch_size, num_heads, query_seq_len, embedding_size // num_heads)

torch.Size([2, 2, 4, 4])

In [23]:
out_proj = torch.nn.Linear(embedding_size, embedding_size)
out_proj = out_proj.to(device)

In [24]:
projected = out_proj(out.transpose(1, 2).reshape(batch_size, query_seq_len, embedding_size))

In [27]:
projected.shape

torch.Size([2, 4, 8])

In [ ]:
q = torch.randn(batch_size, num_heads, query_seq_len, embedding_size, dtype=dtype, device=device)
k = torch.randn(batch_size, num_heads, seq_len, embedding_size, dtype=dtype, device=device)
v = torch.randn(batch_size, num_heads, seq_len, embedding_size, dtype=dtype, device=device)

In [ ]:
with torch.backends.cuda.sdp_kernel(enable_flash=False, enable_math=False, enable_mem_efficient=True):
    out = torch.nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=mask)